In [11]:
import torch


In [12]:
input_size = 1
hidden_size = 16
output_size = 1

In [13]:
wx = torch.randn(input_size, hidden_size) * 0.1
wh = torch.randn(hidden_size, hidden_size) * 0.1
b = torch.randn(hidden_size) * 0.1



In [14]:
wy = torch.randn(hidden_size, output_size) * 0.1
by = torch.zeros(output_size)

for param in [wx, wh, b, wy, by]:
    param.requires_grad = True

In [15]:
def rnn_forward(inputs, h_prev):
    """
    inputs: list of tensors, mỗi tensor shape [input_size]
    h_prev: hidden state ban đầu, shape [hidden_size]

    returns:
        outputs: list các prediction, mỗi cái shape [output_size]
        h:       hidden state cuối cùng, shape [hidden_size]
    """
    h = h_prev
    outputs = []
    for t in range(len(inputs)):
        x_t = inputs[t]

        h = torch.tanh(x_t @ wx + h @ wh + b)

        # output layer
        y_t = h @ wy + by
        outputs.append(y_t)
    return outputs, h


In [16]:
# Test thử với chuỗi [1, 2, 3]
inputs = [torch.tensor([1.0]),
          torch.tensor([2.0]),
          torch.tensor([3.0])]
print(inputs)

h0 = torch.zeros(hidden_size)
print(h0)
outputs, h_final = rnn_forward(inputs, h0)
print(f"outputs: {outputs}")
print(f"h_final: {h_final}")

print("Output tại mỗi bước:", [o.item() for o in outputs])
print("Hidden state cuối:", h_final.shape)


[tensor([1.]), tensor([2.]), tensor([3.])]
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
outputs: [tensor([-0.0056], grad_fn=<AddBackward0>), tensor([-0.0451], grad_fn=<AddBackward0>), tensor([-0.0985], grad_fn=<AddBackward0>)]
h_final: tensor([-0.2710,  0.0277,  0.1138, -0.0171,  0.2083,  0.2685, -0.5819,  0.3539,
        -0.0925, -0.1857,  0.2389, -0.2752, -0.1596, -0.1712,  0.1960, -0.2906],
       grad_fn=<TanhBackward0>)
Output tại mỗi bước: [-0.005580148659646511, -0.045128729194402695, -0.0985088124871254]
Hidden state cuối: torch.Size([16])


In [17]:
data = torch.arange(1, 11, dtype=torch.float32)
seq_len = 3

X, Y = [], []
for i in range(len(data) - seq_len):
    X.append(data[i:i + seq_len])
    Y.append(data[i + seq_len])

print(f"Số samples: {len(X)}")
print(f"Ví dụ X[0]: {X[0]}, Y[0]: {Y[0]}")

Số samples: 7
Ví dụ X[0]: tensor([1., 2., 3.]), Y[0]: 4.0


In [18]:
learning_rate = 0.01
epochs = 200

for epoch in range(epochs):
    total_loss = 0.0

    for i in range(len(X)):
        inputs = [X[i][t].unsqueeze(0) for t in range(seq_len)]
        target = Y[i]

        h0 = torch.zeros(hidden_size)

        outputs, _ = rnn_forward(inputs, h0)

        y_pred = outputs[-1]

        loss = (y_pred - target) ** 2

        loss.backward()

        with torch.no_grad():
            for param in [wx, wh, b, wy, by]:
                param -= learning_rate * param.grad
                param.grad.zero_()

        total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(X):.4f}")

Epoch 50, Loss: 0.2480
Epoch 100, Loss: 0.0484
Epoch 150, Loss: 0.0142
Epoch 200, Loss: 0.0199


In [20]:
import torch.nn as nn

class RNNModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc  = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x) # out shape: [batch, seq_len, hidden_size]
        out = out[:, -1, :]  # lấy bước cuối: [batch, hidden_size]
        return self.fc(out) # [batch, output_size]

model = RNNModel(input_size, hidden_size, output_size=1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()
X_tensor = torch.stack(X).unsqueeze(-1)  # [7, 3, 1]
Y_tensor = torch.stack(Y) if isinstance(Y[0], torch.Tensor) else torch.tensor(Y)  # [7]

for epoch in range(500):
    model.train()
    optimizer.zero_grad()

    y_pred = model(X_tensor).squeeze()   # [7]
    loss   = criterion(y_pred, Y_tensor)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 50, Loss: 0.2480
Epoch 100, Loss: 0.1807
Epoch 150, Loss: 0.2274
Epoch 200, Loss: 0.1953
Epoch 250, Loss: 0.1543
Epoch 300, Loss: 0.1246
Epoch 350, Loss: 0.0992
Epoch 400, Loss: 0.0792
Epoch 450, Loss: 0.0642
Epoch 500, Loss: 0.0531
